# TCG-pokebot — train & evaluate on Kaggle (native Linux)

Kaggle runs Linux x86-64, so the `cabt` engine loads directly — no Docker needed.

**Before running:** add data via the right panel — *Add Input* →
1. the **Pokémon TCG AI Battle** competition (gives the engine `cg/`, `EN_Card_Data.csv`, sample `deck.csv`).
2. (optional) turn on **Internet** so the first cell can `git clone` our code; otherwise upload the repo as a dataset.

This notebook: locates the engine → generates card tables → imports Limitless decklists →
self-play data → trains the behavioral-cloning net → evaluates vs the varied deck pool →
writes `model.pt` and `submission.tar.gz` to `/kaggle/working`.

In [ ]:
import os, sys, glob, subprocess

# 1) get our code
REPO = '/kaggle/working/TCG-pokebot'
if not os.path.exists(REPO):
    try:
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/Hakase-0/TCG-pokebot', REPO], check=True)
    except Exception as e:
        print('git clone failed (no internet?). Upload the repo as a dataset and set REPO.', e)
sys.path.insert(0, REPO)
os.chdir(REPO)

# 2) locate the competition input (engine + card data) and link it in
cands = glob.glob('/kaggle/input/*')
print('inputs:', cands)
comp = next((p for p in cands if glob.glob(os.path.join(p,'**','cg'), recursive=True)
             or glob.glob(os.path.join(p,'**','EN_Card_Data.csv'), recursive=True)), None)
assert comp, 'Attach the competition input (it provides cg/ and EN_Card_Data.csv).'

def _find(name):
    hits = glob.glob(os.path.join(comp,'**',name), recursive=True)
    return hits[0] if hits else None
card_csv = _find('EN_Card_Data.csv')
sample   = _find('deck.csv')
# symlink engine + data into the repo dir
if _find('cg') and not os.path.exists('cg'):
    os.symlink(_find('cg'), os.path.join(REPO,'cg'))
if card_csv and not os.path.exists('EN_Card_Data.csv'):
    os.symlink(card_csv, os.path.join(REPO,'EN_Card_Data.csv'))
if sample and not os.path.exists('deck.csv'):
    import shutil; shutil.copy(sample, os.path.join(REPO,'deck.csv'))
print('cg present:', os.path.exists('cg'), '| card csv:', os.path.exists('EN_Card_Data.csv'))
!pip -q install kaggle-environments==1.30.1 >/dev/null 2>&1
from cg import api; print('engine OK, cards:', len(api.all_card_data()))

In [ ]:
# 3) generate the card capability/attack tables from the engine (one-time)
!CG_LIB_PATH=$PWD/cg python inspect_cards.py
print('tables:', os.path.exists('capability_table.json'), os.path.exists('attack_table.json'))

In [ ]:
# 4) import the Limitless decklists in decks/*.txt into engine deck CSVs
from import_deck import import_decklist, write_deck_csv
for txt in glob.glob('decks/*.txt'):
    ids, rep = import_decklist(open(txt).read(), 'EN_Card_Data.csv')
    out = txt[:-4] + '.csv'
    if rep['legal_60']:
        write_deck_csv(ids, out)
    print(os.path.basename(txt), '->', rep['mapped'], 'mapped, legal60=', rep['legal_60'],
          '' if not rep['unmatched'] else ('UNMATCHED: '+str(rep['unmatched'])))

In [ ]:
# 5) generate self-play data (distill the strong combat agent) and train the BC net
!python gen_selfplay_data.py --games 300 --policy combat --out data/bc.pkl --log logs/selfplay.jsonl
!python train_bc.py --data data/bc.pkl --epochs 20 --out model.pt --log logs/train.jsonl
import stats; print(stats.read('logs/train.jsonl')[-1])

In [ ]:
# 6) evaluate vs the varied imported decks (heuristic vs nn — set POLICY)
!python eval_vs_decks.py --decks decks/ --games 30 --log logs/matchups.jsonl
!python stats_ui.py --train logs/train.jsonl --eval logs/matchups.jsonl 2>/dev/null || true

In [ ]:
# 7) package the submission and copy artifacts to /kaggle/working
import shutil
!CG_LIB_PATH=$PWD/cg bash build_submission.sh
for f in ['model.pt','model_meta.json','submission.tar.gz']:
    if os.path.exists(f): shutil.copy(f, '/kaggle/working/'+f)
print('artifacts in /kaggle/working:', os.listdir('/kaggle/working'))